# Text2Graph improvement

In [33]:
text_sample = """
When a small hobbit named Frodo Baggins inherits a magic ring from his uncle, the wizard Gandalf investigates and discovers that the ring is an ancient creation of an evil dark lord. Should the ring end up back in his hands, he will regain his power and destroy Middle Earth. Frodo and his loyal friends set out on a a quest to destroy the ring with a band of warriors. This is an underrated adaption of the classic novels as it only covers the first half of the story. Regardless, this is an epic and wonderfully animated film.<br /><br />The animation is superbly done with rotoscope, which is tracing over live action footage. Ralph Bakshi worked well with the low budget he was given. The film also boasts a grand music score by Leonard Rosenman that fits every scene. There are a few plot holes with the script, but that has to be excused, considering the original deal was to make the book trilogy into two films, much had to crammed in the first one. My biggest gripe is some of the character design, Samwise was a bit too goofy, while the other hobbits act completely normal. The other characters are actually well written for the screen, and the voice actors do a great job, I was pleased that Legolas is actually a bit more helpful to the plot. The rotoscoped orcs are more comical than frightening, while the ringwraiths are eerie and nightmarish.<br /><br />Another problem is that the evil wizard Saruman is called Aruman, thanks to the writers. Overall, I think a little more money and better writers would have done this a lot of justice, but there is something charming about it. Ralph Bakshi made a valiant effort of making screen adaption of these classic stories. The film suffers terribly from being overshadowed by the live action films, but it's still a great movie for animation lovers of all ages.
"""

In [34]:
import re
import logging
from collections import defaultdict, Counter
from math import log
from typing import List, Tuple

logger = logging.getLogger("TextProcessor.SmartNGramTokenizer")

class SmartNGramTokenizer:
    def __init__(self, ngram_range: Tuple[int, int] = (2, 3), lowercase: bool = True, pmi_threshold: float = 3.0):
        """
        Initializes the Smart N-Gram Tokenizer with dynamic n-gram discovery using PMI.

        Args:
            ngram_range (Tuple[int, int]): Range of n-grams to generate (default: (2, 3)).
            lowercase (bool): Whether to convert text to lowercase (default: True).
            pmi_threshold (float): Minimum PMI score for a meaningful n-gram (default: 3.0).
        """
        self.ngram_range = ngram_range
        self.lowercase = lowercase
        self.pmi_threshold = pmi_threshold
        logger.info(f"SmartNGramTokenizer initialized with ngram_range={ngram_range}, PMI threshold={pmi_threshold}, lowercase={lowercase}")

    def clean_text(self, text: str) -> str:
        """
        Cleans the input text by removing unwanted characters.

        Args:
            text (str): Raw text to clean.

        Returns:
            str: Cleaned text.
        """
        if not text:
            logger.warning("Received an empty text for cleaning.")
            return ""
        
        logger.debug("Cleaning text...")
        if self.lowercase:
            text = text.lower()
        
        # Remove unwanted characters
        cleaned_text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
        cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
        logger.debug(f"Cleaned text: {cleaned_text}")
        return cleaned_text

    def tokenize(self, text: str) -> List[str]:
        """
        Tokenizes the cleaned text into words and discovers meaningful n-grams using PMI.

        Args:
            text (str): Cleaned text to tokenize.

        Returns:
            List[str]: List of tokens (words and meaningful n-grams).
        """
        if not text:
            logger.warning("Received an empty text for tokenization.")
            return []

        logger.debug("Tokenizing text...")
        words = text.split()
        tokens = words[:]  # Always keep single words
        meaningful_ngrams = self._calculate_pmi(words)
        tokens.extend(meaningful_ngrams)
        logger.debug(f"Tokens: {tokens}")
        return tokens

    def _calculate_pmi(self, words: List[str]) -> List[str]:
        """
        Calculates PMI for all n-grams in the text and identifies meaningful ones.

        Args:
            words (List[str]): List of words in the text.

        Returns:
            List[str]: List of meaningful n-grams.
        """
        logger.debug("Calculating PMI for n-grams...")
        total_count = len(words)
        word_counts = Counter(words)
        ngram_counts = defaultdict(int)

        # Sliding window for n-grams
        for n in range(self.ngram_range[0], self.ngram_range[1] + 1):
            for i in range(len(words) - n + 1):
                ngram = " ".join(words[i:i + n])
                ngram_counts[ngram] += 1

        meaningful_ngrams = []

        # Calculate PMI for each n-gram
        for ngram, count in ngram_counts.items():
            if count < 2:  # Ignore rare n-grams
                continue
            
            ngram_words = ngram.split()
            p_ngram = count / total_count
            p_individual = 1
            for word in ngram_words:
                p_individual *= (word_counts[word] / total_count)

            # PMI calculation
            pmi = log(p_ngram / p_individual)
            logger.debug(f"PMI({ngram}) = {pmi:.4f}")

            if pmi >= self.pmi_threshold:
                meaningful_ngrams.append(ngram)

        logger.info(f"Meaningful n-grams discovered: {meaningful_ngrams}")
        return meaningful_ngrams


In [35]:
import logging
from typing import List, Tuple
import spacy

logger = logging.getLogger("TextProcessor.POSTagger")

class POSTagger:
    def __init__(self):
        """
        Initializes the POS Tagger.
        """
        logger.info("POSTagger initialized.")

    def tag_tokens(self, tokens: List[str], nlp_model: spacy.Language) -> List[Tuple[str, str]]:
        """
        Tags each token (single word or n-gram) with its POS tag.

        Args:
            tokens (List[str]): List of tokens (words or n-grams).
            nlp_model (spacy.Language): SpaCy NLP model for POS tagging.

        Returns:
            List[Tuple[str, str]]: List of (token, POS) pairs.
        """
        logger.info("Tagging tokens with POS...")
        tagged_tokens = []

        for token in tokens:
            if " " in token:  # Multi-word n-gram
                pos = self._tag_ngram(token, nlp_model)
            else:  # Single word
                doc = nlp_model(token)
                pos = doc[0].pos_ if doc else "UNKNOWN"
            
            tagged_tokens.append((token, pos))
            logger.debug(f"Tagged: {token} -> {pos}")

        logger.info("POS tagging completed.")
        return tagged_tokens

    def _tag_ngram(self, ngram: str, nlp_model: spacy.Language) -> str:
        """
        Tags a multi-word n-gram using SpaCy.

        Args:
            ngram (str): The n-gram to tag.
            nlp_model (spacy.Language): SpaCy NLP model for POS tagging.

        Returns:
            str: POS tag for the n-gram.
        """
        logger.debug(f"Tagging n-gram: {ngram}")

        # Attempt to directly tag the entire n-gram
        doc = nlp_model(ngram)
        if len(doc) == 1:
            return doc[0].pos_  # Direct POS if recognized as a single entity

        # If n-gram is not recognized, derive POS from its components
        pos_counts = {}
        for token in doc:
            if token.pos_ in pos_counts:
                pos_counts[token.pos_] += 1
            else:
                pos_counts[token.pos_] = 1
        
        # Use the most frequent POS among the components
        if pos_counts:
            most_frequent_pos = max(pos_counts, key=pos_counts.get)
            logger.debug(f"N-gram '{ngram}' POS derived from majority: {most_frequent_pos}")
            return most_frequent_pos

        logger.warning(f"Failed to tag n-gram: {ngram}. Marked as UNKNOWN.")
        return "UNKNOWN"


In [36]:
import spacy
import logging
from typing import List, Tuple, Set

logger = logging.getLogger("TextProcessor.DependencyParser")

class DependencyParser:
    def __init__(self, nlp_model: spacy.Language, exclude_deps: Set[str] = {"punct", "det", "aux", "cc"}):
        """
        Initializes the Dependency Parser with a SpaCy model and exclusion list.

        Args:
            nlp_model (spacy.Language): The SpaCy NLP model for dependency parsing.
            exclude_deps (Set[str]): Set of dependency types to exclude (default: {"punct", "det", "aux", "cc"}).
        """
        self.nlp = nlp_model
        self.exclude_deps = exclude_deps
        logger.info(f"DependencyParser initialized with SpaCy model and exclusions: {exclude_deps}")

    def set_excluded_dependencies(self, deps: Set[str]) -> None:
        """
        Sets a custom list of excluded dependency types.

        Args:
            deps (Set[str]): Set of dependency types to exclude.
        """
        self.exclude_deps = deps
        logger.info(f"Excluded dependencies updated: {deps}")

    def parse(self, text: str, tokens: List[str]) -> List[Tuple[str, str, str]]:
        """
        Parses the text to extract dependency relations.

        Args:
            text (str): The raw text to parse.
            tokens (List[str]): The list of tokens (words or n-grams).

        Returns:
            List[Tuple[str, str, str]]: List of (head, dependent, relation).
        """
        if not text:
            logger.warning("Received an empty text for dependency parsing.")
            return []

        logger.debug("Parsing text for dependencies...")
        doc = self.nlp(text)
        dependencies = []

        # Detect and merge multi-word n-grams in the dependency tree
        ngram_map = self._merge_ngrams(tokens)

        for token in doc:
            # Skip excluded dependencies
            if token.dep_ in self.exclude_deps:
                logger.debug(f"Excluding dependency: {token.text} ({token.dep_})")
                continue

            # If the token is part of an n-gram, use the n-gram instead
            head = ngram_map.get(token.head.text, token.head.text)
            dependent = ngram_map.get(token.text, token.text)
            relation = token.dep_

            if head != dependent:  # Avoid self-loops
                dependencies.append((head, dependent, relation))
                logger.debug(f"Dependency: {head} -> {dependent} ({relation})")

        logger.info("Dependency parsing completed.")
        return dependencies

    def _merge_ngrams(self, tokens: List[str]) -> dict:
        """
        Merges multi-word n-grams into single tokens for dependency parsing.

        Args:
            tokens (List[str]): The list of tokens (including n-grams).

        Returns:
            dict: Mapping of words to their merged n-grams.
        """
        logger.debug("Merging multi-word n-grams in dependency tree...")
        ngram_map = {}

        # Find n-grams in the list of tokens
        for ngram in tokens:
            if " " in ngram:  # Multi-word n-gram
                ngram_words = ngram.split()
                # Map each word in the n-gram to the full n-gram
                for word in ngram_words:
                    ngram_map[word] = ngram
        
        return ngram_map


In [37]:
import networkx as nx
import logging
from typing import List, Tuple

logger = logging.getLogger("TextProcessor.GraphBuilder")

class GraphBuilder:
    def __init__(self):
        """
        Initializes the GraphBuilder.
        """
        logger.info("GraphBuilder initialized.")

    def add_nodes(self, graph: nx.DiGraph, tokens: List[Tuple[str, str]]) -> None:
        """
        Adds tokens as nodes in the graph.

        Args:
            graph (nx.DiGraph): The directed graph to modify.
            tokens (List[Tuple[str, str]]): List of (token, POS) pairs.
        """
        logger.info("Adding nodes to the graph...")
        for token, pos in tokens:
            if token not in graph:
                graph.add_node(token, pos=pos, type="n-gram" if " " in token else "word")
                logger.debug(f"Added node: {token} (POS: {pos})")

    def add_edges(self, graph: nx.DiGraph, dependencies: List[Tuple[str, str, str]]) -> None:
        """
        Adds directed edges based on dependencies.

        Args:
            graph (nx.DiGraph): The directed graph to modify.
            dependencies (List[Tuple[str, str, str]]): List of (head, dependent, relation).
        """
        logger.info("Adding edges to the graph...")
        for head, dependent, relation in dependencies:
            if head in graph and dependent in graph:
                # If the edge already exists, increase weight
                if graph.has_edge(head, dependent):
                    graph[head][dependent]['weight'] += 1
                else:
                    graph.add_edge(head, dependent, relation=relation, weight=1)
                logger.debug(f"Added edge: {head} -> {dependent} (Relation: {relation})")
            else:
                logger.warning(f"Edge ignored: {head} -> {dependent} (Nodes not found in graph)")


In [38]:
import spacy
import networkx as nx
import logging
from typing import Optional, Tuple, Set

logger = logging.getLogger("TextProcessor")

class TextProcessor:
    def __init__(
        self,
        language: str = "en_core_web_lg",
        ngram_range: Tuple[int, int] = (2, 3),
        pmi_threshold: float = 3.0,
        exclude_deps: Set[str] = {"punct", "det", "aux", "cc"}
    ):
        """
        Initializes the TextProcessor with a language model, smart tokenizer, and POS tagger.

        Args:
            language (str): SpaCy language model to use (default: "en_core_web_sm").
            ngram_range (Tuple[int, int]): Range of n-grams for meaningful discovery (default: (2, 3)).
            pmi_threshold (float): PMI threshold for meaningful n-grams (default: 3.0).
            exclude_deps (Set[str]): Dependency types to exclude (default: {"punct", "det", "aux", "cc"}).
        """
        logger.info("Initializing TextProcessor...")
        self.nlp = spacy.load(language)
        self.graph = None  # Graph will be initialized for each document
        self.raw_text = ""
        
        # Initialize Smart N-Gram Tokenizer, POS Tagger, Dependency Parser, and GraphBuilder
        self.tokenizer = SmartNGramTokenizer(ngram_range=ngram_range, pmi_threshold=pmi_threshold)
        self.pos_tagger = POSTagger()
        self.dependency_parser = DependencyParser(self.nlp, exclude_deps=exclude_deps)
        self.graph_builder = GraphBuilder()
        logger.info(f"TextProcessor initialized with model: {language}")

    def set_excluded_dependencies(self, deps: Set[str]) -> None:
        """
        Sets a custom list of excluded dependency types in the DependencyParser.

        Args:
            deps (Set[str]): Set of dependency types to exclude.
        """
        self.dependency_parser.set_excluded_dependencies(deps)
        logger.info(f"Excluded dependencies updated in DependencyParser: {deps}")

    def process_text(self, text: str) -> None:
        """
        Processes a given text through the pipeline:
        - Tokenization and Cleaning with Smart N-Gram Discovery
        - POS Tagging with POSTagger
        - Dependency Parsing
        - Graph Building

        Args:
            text (str): The raw text to process.
        """
        logger.info("Processing new document...")
        self.raw_text = text
        self.graph = nx.DiGraph()  # Reset the graph for each document
        
        # Text Processing Pipeline
        logger.debug("Cleaning and tokenizing text using SmartNGramTokenizer...")
        cleaned_text = self.tokenizer.clean_text(self.raw_text)
        tokens = self.tokenizer.tokenize(cleaned_text)
        logger.debug(f"Tokens: {tokens}")

        logger.debug("Tagging tokens with POS using POSTagger...")
        tagged_tokens = self.pos_tagger.tag_tokens(tokens, self.nlp)
        logger.debug(f"Tagged Tokens: {tagged_tokens}")

        logger.debug("Extracting dependencies using DependencyParser...")
        dependencies = self.dependency_parser.parse(cleaned_text, tokens)
        logger.debug(f"Dependencies: {dependencies}")

        logger.debug("Building graph using GraphBuilder...")
        self.graph_builder.add_nodes(self.graph, tagged_tokens)
        self.graph_builder.add_edges(self.graph, dependencies)
        logger.info("Graph successfully built for the document.")

    def get_graph(self) -> nx.DiGraph:
        """
        Returns the processed text graph.

        Returns:
            nx.DiGraph: The directed graph of the processed text.
        """
        if self.graph is None:
            logger.error("No graph has been created. Please process a text first.")
            raise ValueError("No graph has been created. Please process a text first.")
        logger.info("Graph returned successfully.")
        return self.graph

    def get_text(self) -> str:
        """
        Returns the cleaned, processed text.

        Returns:
            str: The processed text.
        """
        return self.raw_text


In [ ]:
import networkx as nx
import plotly.graph_objects as go
import logging

logger = logging.getLogger("GraphVisualizer")

class GraphVisualizer:
    def __init__(self, graph: nx.DiGraph):
        """
        Initializes the GraphVisualizer with a NetworkX graph.

        Args:
            graph (nx.DiGraph): The directed graph to visualize.
        """
        self.graph = graph
        self.layout = "spring"
        self.node_color_map = {"word": "skyblue", "n-gram": "lightgreen"}
        self.edge_color = "gray"
        self.node_size = 15
        self.edge_width = 1.0
        logger.info("GraphVisualizer initialized.")

    def set_layout(self, layout: str = "spring") -> None:
        """
        Sets the layout for the graph visualization.

        Args:
            layout (str): The layout type ("spring", "circular", "random").
        """
        if layout not in {"spring", "circular", "random"}:
            logger.warning(f"Unknown layout '{layout}', using default (spring).")
            layout = "spring"
        self.layout = layout
        logger.info(f"Graph layout set to: {layout}")

    def customize(
        self,
        node_color_map: dict = None,
        edge_color: str = "gray",
        node_size: int = 15,
        edge_width: float = 1.0
    ) -> None:
        """
        Customizes the appearance of nodes and edges.

        Args:
            node_color_map (dict): Map of node types to colors.
            edge_color (str): Color of the edges.
            node_size (int): Size of the nodes.
            edge_width (float): Width of the edges.
        """
        if node_color_map:
            self.node_color_map = node_color_map
        self.edge_color = edge_color
        self.node_size = node_size
        self.edge_width = edge_width
        logger.info("Graph appearance customized.")

    def show(self) -> None:
        """
        Displays the interactive graph using Plotly.
        """
        # Select layout
        if self.layout == "spring":
            pos = nx.spring_layout(self.graph, seed=42)
        elif self.layout == "circular":
            pos = nx.circular_layout(self.graph)
        elif self.layout == "random":
            pos = nx.random_layout(self.graph)

        # Extracting node data
        node_x, node_y, node_text, node_color = [], [], [], []

        for node, attrs in self.graph.nodes(data=True):
            x, y = pos[node]
            node_x.append(x)
            node_y.append(y)
            node_text.append(f"{node} ({attrs.get('pos', 'N/A')})")
            node_color.append(self.node_color_map.get(attrs.get("type", "word"), "skyblue"))

        # Extracting edge data
        edge_x, edge_y, edge_text = [], [], []

        for source, target, attrs in self.graph.edges(data=True):
            x0, y0 = pos[source]
            x1, y1 = pos[target]
            edge_x += [x0, x1, None]
            edge_y += [y0, y1, None]
            edge_text.append(f"{source} → {target} ({attrs.get('relation', 'N/A')})")

        # Creating the figure
        fig = go.Figure()

        # Adding edges
        fig.add_trace(go.Scatter(
            x=edge_x, y=edge_y,
            line=dict(width=self.edge_width, color=self.edge_color),
            hoverinfo='text',
            text=edge_text,
            mode='lines'
        ))

        # Adding nodes
        fig.add_trace(go.Scatter(
            x=node_x, y=node_y,
            mode='markers+text',
            marker=dict(
                size=self.node_size,
                color=node_color,
                line=dict(width=2, color='darkgray')
            ),
            text=node_text,
            textposition="top center",
            hoverinfo='text'
        ))

        # Customizing layout
        fig.update_layout(
            title="Interactive Dependency Graph",
            showlegend=False,
            hovermode="closest",
            margin=dict(l=0, r=0, t=40, b=0),
            width=800,
            height=600
        )

        fig.show()

    def save_html(self, file_name: str) -> None:
        """
        Saves the interactive graph as an HTML file.

        Args:
            file_name (str): The name of the HTML file to save.
        """
        logger.info(f"Saving graph to {file_name}...")
        fig = self.show()
        fig.write_html(file_name)
        logger.info(f"Graph saved as {file_name}.")
